# 🎨 Manga Studio — Colorizador Automático de Mangás com IA (Google Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jos9011/SKILL-TRADUCAO-MANGA/blob/main/Colab_Colorir_Manga.ipynb)

Este notebook implementa a **Abordagem A (Colorização em Lote via Google Colab)** do **Manga Studio**:
- ⚡ **100% Gratuito & Acelerado por GPU (T4 / V100 / A100)**: Não consome processador nem memória do seu PC.
- 🚀 **Velocidade Ultra-Rápida**: Processa cada página em ~0.5s a 1.5s na GPU do Colab.
- 🖋️ **Preservação Cirúrgica do Nanquim**: Mantém os traços pretos originais 100% nítidos sem borrões.
- 📁 **Lê e Salva Direto no Google Drive**: Funciona para mangás originais P&B ou já traduzidos em `MANGAS [PT-BR]`.
- 📦 **Exportação Opcional em .CBZ**: Cria pacote pronto para leitores como Mihon, Tachiyomi, CDisplayEx ou Kindle.

In [ ]:
#@title 🚀 Passo 1: Verificar Aceleração de Hardware (GPU)
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✅ GPU Ativa: {gpu_name}')
    print('✨ Aceleração por hardware CUDA pronta para colorização ultra-rápida!')
else:
    print('⚠️ Nenhuma GPU detectada.')
    print("👉 Vá no menu superior: 'Ambiente de Execução' -> 'Alterar tipo de ambiente de execução' e selecione 'T4 GPU'.")

In [ ]:
#@title 📁 Passo 2: Montar Google Drive e Instalar Dependências
from google.colab import drive
import os

print('[*] Conectando ao Google Drive...')
drive.mount('/content/drive')
print('✅ Google Drive conectado com sucesso!')

print('\n[*] Instalando bibliotecas neurais (ONNX Runtime GPU, OpenCV, Pillow)...')
!pip install -q onnxruntime-gpu opencv-python pillow tqdm matplotlib
print('✅ Todas as dependências foram instaladas com sucesso!')

In [ ]:
#@title 🧠 Passo 3: Baixar Modelo Neural Otimizado (~58MB)
import os, urllib.request

os.makedirs('/content/models', exist_ok=True)
MODEL_PATH = '/content/models/manga-colorize-fp16.onnx'
MODEL_URL = 'https://huggingface.co/Faridzar/manga-colorization-v2-onnx/resolve/main/manga-colorize-fp16.onnx'

if not os.path.exists(MODEL_PATH) or os.path.getsize(MODEL_PATH) < 10*1024*1024:
    print('[*] Baixando modelo neural de colorização ONNX FP16...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(f'✅ Modelo pronto em: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/(1024*1024):.1f} MB)')
else:
    print(f'✅ Modelo já carregado: {MODEL_PATH}')

In [ ]:
#@title ⚙️ Passo 4: Inicializar Motor de Inferência na GPU
import cv2, re, time, zipfile
import numpy as np
import onnxruntime as ort
from PIL import Image
from tqdm import tqdm

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']
session_opts = ort.SessionOptions()
session_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
color_session = ort.InferenceSession(MODEL_PATH, sess_options=session_opts, providers=providers)
print(f'✅ Sessão ONNX iniciada com o provedor: [{color_session.get_providers()[0]}]')

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def colorize_cv2_image(img_bgr, max_size=None, preserve_lineart=True):
    orig_h, orig_w = img_bgr.shape[:2]
    orig_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    work = img_bgr
    scale = 1.0
    if max_size and max_size > 0 and max(orig_h, orig_w) > max_size:
        scale = max_size / float(max(orig_h, orig_w))
        nw, nh = max(32, int(round(orig_w * scale))), max(32, int(round(orig_h * scale)))
        work = cv2.resize(img_bgr, (nw, nh), interpolation=cv2.INTER_AREA)
    h, w = work.shape[:2]
    pad_h = (32 - (h % 32)) % 32
    pad_w = (32 - (w % 32)) % 32
    gray_norm = cv2.cvtColor(work, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    if pad_h > 0 or pad_w > 0:
        gray_norm = cv2.copyMakeBorder(gray_norm, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
    th, tw = gray_norm.shape
    inp = np.zeros((1, 5, th, tw), dtype=np.float32)
    inp[0, 0, :, :] = gray_norm
    out = color_session.run(['rgb'], {'input': inp})[0][0, :, :h, :w]
    out = np.transpose(out, (1, 2, 0))
    out_bgr = np.clip(out * 255.0, 0, 255).astype(np.uint8)
    out_bgr = cv2.cvtColor(out_bgr, cv2.COLOR_RGB2BGR)
    if scale != 1.0:
        out_bgr = cv2.resize(out_bgr, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)
    if preserve_lineart:
        ycrcb = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2YCrCb)
        ycrcb[:, :, 0] = orig_gray
        out_bgr = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)
    return out_bgr

def make_cbz_archive(folder_path, cbz_path):
    with zipfile.ZipFile(cbz_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        files = sorted(os.listdir(folder_path), key=natural_sort_key)
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                zf.write(os.path.join(folder_path, f), arcname=f)
    print(f'📦 Pacote CBZ gerado: {cbz_path} ({os.path.getsize(cbz_path)/(1024*1024):.1f} MB)')

In [ ]:
#@title 🎨 Passo 5: Colorir Obra do Google Drive
#@markdown Digite o caminho da pasta no seu Drive e ajuste as preferências:

PASTA_ENTRADA = "/content/drive/MyDrive/MANGAS [PT-BR]/Minha Obra [PT-BR]" #@param {type:"string"}
PASTA_SAIDA = "" #@param {type:"string"}
GERAR_CBZ = True #@param {type:"boolean"}
PRESERVAR_NANQUIM = True #@param {type:"boolean"}
MAX_DIMENSAO = 0 #@param {type:"integer"}

input_dir = PASTA_ENTRADA.strip('"\'')
if not os.path.isdir(input_dir):
    raise FileNotFoundError(f'Pasta não encontrada: {input_dir}. Verifique o caminho no Google Drive!')

exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
img_names = [f for f in os.listdir(input_dir) if f.lower().endswith(exts)]
img_names.sort(key=natural_sort_key)
if not img_names:
    raise ValueError(f'Nenhuma imagem encontrada na pasta: {input_dir}')

if not PASTA_SAIDA.strip():
    base_name = os.path.basename(input_dir.rstrip('/\\'))
    clean_n = base_name.replace(' [COLORIDO]', '').replace('[COLORIDO]', '').strip()
    parent_dir = os.path.dirname(input_dir.rstrip('/\\'))
    output_dir = os.path.join(parent_dir, f'{clean_n} [COLORIDO]')
else:
    output_dir = PASTA_SAIDA.strip('"\'')

os.makedirs(output_dir, exist_ok=True)
print('=' * 60)
print(f'[*] INICIANDO COLORIZACAO DE {len(img_names)} PAGINAS')
print(f'[*] Entrada: {input_dir}')
print(f'[*] Saida:   {output_dir}')
print('=' * 60)

t0 = time.time()
sample_preview_orig = None
sample_preview_color = None

for idx, fname in enumerate(tqdm(img_names, desc='Colorindo Paginas', unit='pag')):
    in_file = os.path.join(input_dir, fname)
    bname, _ = os.path.splitext(fname)
    out_file = os.path.join(output_dir, f'{bname}.jpg')
    with open(in_file, 'rb') as f:
        buf = np.asarray(bytearray(f.read()), dtype=np.uint8)
        img_bgr = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    if img_bgr is None:
        continue
    colored = colorize_cv2_image(img_bgr, max_size=MAX_DIMENSAO or None, preserve_lineart=PRESERVAR_NANQUIM)
    is_ok, enc = cv2.imencode('.jpg', colored, [cv2.IMWRITE_JPEG_QUALITY, 95])
    if is_ok:
        with open(out_file, 'wb') as f:
            f.write(enc.tobytes())
    if sample_preview_orig is None:
        sample_preview_orig = img_bgr
        sample_preview_color = colored

total_time = time.time() - t0
per_page = total_time / max(1, len(img_names))
print('\n' + '=' * 60)
print(f'✅ Colorizacao concluida em {total_time:.1f}s ({per_page:.2f}s por pagina)!')
print(f'📁 Pasta com imagens salvas: {output_dir}')

if GERAR_CBZ:
    cbz_name = os.path.basename(output_dir.rstrip('/\\')) + '.cbz'
    cbz_full = os.path.join(os.path.dirname(output_dir), cbz_name)
    make_cbz_archive(output_dir, cbz_full)

# Visualizacao Comparativa Antes x Depois
if sample_preview_orig is not None and sample_preview_color is not None:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(14, 9))
    axes[0].imshow(cv2.cvtColor(sample_preview_orig, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original (P&B)', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(sample_preview_color, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Colorizado com IA (GPU)', fontsize=14, fontweight='bold', color='#1db954')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()